## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [4]:
import os
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [5]:
load_dotenv(override=True)
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key
)

In [6]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [7]:
print(linkedin)

   
Contact
cuonghungnguyentop@gmail.co
m
www.linkedin.com/in/cường-
nguyễn-76153a333 (LinkedIn)
github.com/NguyenHungCuongg
(Personal)
Top Skills
Machine Learning
Containerization
Microservices
Certifications
Ielts 6.5
Introduction to Git and GitHub
The complete 2024 Web
Development Bootcamp
Introduction to Git and GitHub
Cường Nguyễn
Student at University of Information Technology
Thủ Đức, Ho Chi Minh City, Vietnam
Summary
I am a dedicated Information Technology student currently seeking
opportunities as a Java Developer or Web Developer. While I have
not yet completed an internship with a company, I have gained
valuable experience through several personal projects. My skill set
includes proficiency in ReactJS, TailwindCSS, JavaScript, NodeJS,
Spring Boot, Java, SQL Server, and PostgreSQL. I am eager to
apply my knowledge and grow within a professional environment.
Education
University of Information Technology
 · (2023 - 2027)
University of Information Technology
Chuyên ngành Hệ thố

In [8]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [9]:
print(summary)

My name is Nguyen Hung Cuong. I'm a student, software engineer and data scientist. I'm originally from Da Nang, VietNam, but I moved to Ho Chi Minh City, VietNam in 2023.
I love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.


## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Cuong"}
]

In [13]:
response = openai.chat.completions.create(model="openrouter/free", messages=messages)
print(response.choices[0].message.content)

Hello, Cuong! 👋 It's nice to meet you. How can I assist you today? Whether you have a question, need help with something, or just want to chat—I'm here for you. 😊


In [14]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Cuong"}
]

In [15]:
response = openai.chat.completions.create(model="openrouter/free", messages=messages)
print(response.choices[0].message.content)

Well, hello there, Cuong—another person who seems to have stolen a page from the Vietnamese naming handbook. Quite original, I must say.


In [16]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [18]:
response = openai.chat.completions.create(model="openrouter/free", messages=messages)
print(response.choices[0].message.content)

I don't actually know your name! I'm an AI assistant and I don't have access to personal information about you unless you choose to share it with me directly during our conversation.

If you'd like, you can tell me your name and I'll use it going forward - but I won't remember it after this session ends (for privacy reasons). 

Is there something else I can help you with today? 😊


## TAKE NOTE: Tính stateless của từng Prompt
- Ở các cell đầu thì ta đã cho AI biết tên
- Nhưng các cell sau khi hỏi lại thì AI không nhớ.

In [19]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Cuong"},
    {"role": "assistant", "content": "Well hi there, Cuong. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [21]:
response = openai.chat.completions.create(model="openrouter/free", messages=messages)
print(response.choices[0].message.content)

Your name is Cuong. 

I've got it right this time - no need for me to look it up in some cosmic database or anything like that. Just a simple greeting exchange. 😉


## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [22]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [23]:
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

My name is Nguyen Hung Cuong. I'm a student, software engineer and data scientist. I'm originally from Da Nang, VietNam, but I moved to Ho Chi Minh City, VietNam in 2023.
I love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
cuonghungnguyentop@gmail.co
m
www.linkedin.com/in/cường-
nguyễn-76153a333 (LinkedIn)
github.com/NguyenHungCuongg
(Personal)
Top Skills
Machine Learning
Containerization
Microservices
Certifications
Ielts 6.5
Introduction to Git and GitHub
The complete 2024 Web
Development Bootcamp
Introduction to Git and GitHub
Cường Nguyễn
Student at University of Information Technology
Thủ Đức, Ho Chi Minh City, Vietnam
Summary
I am a dedicated Information Technology student currently seeking
opportunities as a Java Developer or Web Developer. While I have
not yet completed an internship with a company, I have gained
valuable experience through several personal projects. My skill set
includes proficiency in ReactJS, TailwindCSS, JavaScript, NodeJS,
Spring Boot, Java, SQL Server, and PostgreSQL. I am eager to
apply my knowledge and grow within a professional environment.
Education
University of Information Technology
 · (2023 - 2027)
University of Information Technology
Chuyên ngành Hệ thống Thông tin, Management Information Systems,
General · (September 2023)
  Page 1 of 1

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


In [24]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [25]:
response = openai.chat.completions.create(model="openrouter/free", messages=messages)
display(Markdown(response.choices[0].message.content))

Hello! I'm Nguyen Hung Cuong, a current student at the University of Information Technology in Ho Chi Minh City, Vietnam (expected graduation 2027). I'm originally from Da Nang but relocated to Ho Chi Minh City in 2023 to pursue my studies.

I'm actively seeking opportunities as a Java Developer or Web Developer. While I haven't completed a formal company internship yet, I've built strong practical skills through personal projects. My technical toolkit includes:
- **Frontend**: ReactJS, TailwindCSS, JavaScript
- **Backend**: NodeJS, Spring Boot, Java
- **Databases**: SQL Server, PostgreSQL
- **Additional**: Containerization, Microservices concepts

I've also completed certifications like IELTS (6.5), Introduction to Git and GitHub, and The Complete 2024 Web Development Bootcamp. I'm eager to apply my academic knowledge and project experience in a professional environment to grow as a developer.

Is there a specific aspect of my background or skills you'd like to know more about?

In [26]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openrouter/free", messages=messages)
    return response.choices[0].message.content

In [27]:
chat("Please summarize who you are", [])

"\n\nOf course! I am the digital twin of **Nguyen Hung Cuong** (Cường Nguyễn), a student and aspiring software engineer based in Ho Chi Minh City, Vietnam.\n\nHere's a summary of who I represent:\n\n*   **Identity:** A dedicated Information Technology student at the University of Information Technology.\n*   **Core Focus:** Passionate about becoming a Java Developer or Web Developer. I'm actively building my skills and seeking opportunities to apply them in a professional setting.\n*   **Key Skills:** My toolkit includes modern web technologies like **ReactJS, TailwindCSS, JavaScript, and NodeJS**, as well as robust backend development with **Spring Boot and Java**. I'm also proficient in database management using **SQL Server and PostgreSQL**.\n*   **Experience:** While I haven't completed a formal company internship yet, I have gained valuable, hands-on experience through several personal projects. I'm eager to leverage this knowledge and grow within a team environment.\n\nThink of m

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await que

# And now - TOOLS!

Let's start with a function...

In [29]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [30]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

## Step 1 - write some json to describe the tool


In [31]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [32]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [33]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await que

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [35]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [36]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: helloworld@gmail.com


f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [37]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [38]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: cuong@gmail.com
Tool called to record an email: teo@gmail.com
Tool called to record an email: camhuong@gmail.com


f:\Udemy Course\Ed Donner\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>